In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from model.autoencoder import Model
from train import Trainer
from data import get_cross_data, get_rec_data, load_data

In [2]:
batch_size = 32
N = batch_size  # Batch size
T = 64          # Number of frames (64)
M = 1           # Number of persons
V = 25          # Number of joints
C_in = 3        # Number of input channels
setting = 'cs'  # 'cs' or 'cv'
dataset = 'ntu120'

In [3]:
X = load_data(dataset)
paired_train, paired_test = get_cross_data(X, dataset, setting, batch_size, T, return_loader=True, train_samples=64, test_samples=32)#train_samples=50016, test_samples=5024)
train, test = get_rec_data(X, dataset, setting, T, batch_size,)

In [4]:
# Initialize the model
model = Model(num_class=120, num_point=25, num_person=1, graph='graph.ntu_rgb_d.Graph',
              graph_args={'labeling_mode': 'spatial'}, debug=False)
model = model.cuda()

# Define optimizer and loss criterion
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# Number of epochs for each stage
num_epochs_stage1 = 1
num_epochs_stage2 = 1

# Create Trainer instance
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train,
    val_loader=test,
    train_paired_loader=paired_train,
    val_paired_loader=paired_test,
    num_epochs_stage1=num_epochs_stage1,
    num_epochs_stage2=num_epochs_stage2,
    device='cuda'  # or 'cpu' if not using GPU
)

# Train Stage 1
trainer.train_stage1()

# Train Stage 2
trainer.train_stage2()

e:\LocalCode\Skeleton-MixFormer-main\model\autoencoder.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.encoder.load_state_dict(torch.load(path), strict=False)


RuntimeError: Error(s) in loading state_dict for Encoder:
	size mismatch for data_bn.weight: copying a param with shape torch.Size([4000]) from checkpoint, the shape in current model is torch.Size([2000]).
	size mismatch for data_bn.bias: copying a param with shape torch.Size([4000]) from checkpoint, the shape in current model is torch.Size([2000]).
	size mismatch for data_bn.running_mean: copying a param with shape torch.Size([4000]) from checkpoint, the shape in current model is torch.Size([2000]).
	size mismatch for data_bn.running_var: copying a param with shape torch.Size([4000]) from checkpoint, the shape in current model is torch.Size([2000]).